In [1]:
import pandas as pd
import numpy as np
import requests
import time

# -------------------------------
# CONFIG
# -------------------------------
INPUT_FILE = "AU Library Books -ISBN-4-7-68 (1).csv"
OUTPUT_FILE = "books_with_metadata.csv"
N_BOOKS = 10
API_URL = "https://www.googleapis.com/books/v1/volumes"

# -------------------------------
# STEP 1: Load raw data
# -------------------------------
df = pd.read_csv(INPUT_FILE, encoding="latin1")

# -------------------------------
# STEP 2: Clean data
# -------------------------------
def clean_text(x):
    if pd.isna(x):
        return np.nan
    return str(x).strip().replace("_", "").replace('"', "").replace("'", "").strip()

def fix_year(x):
    try:
        year = int(float(x))
        if year > 2025:  # Thai Buddhist year → convert
            year -= 543
        return year
    except:
        return np.nan

def clean_isbn(x):
    if pd.isna(x):
        return np.nan
    return str(x).replace("-", "").replace(" ", "")

# Apply cleaning
for col in ["Author", "Title", "Edition", "Imprint", "Call No.", "ISBN"]:
    df[col] = df[col].apply(clean_text)

df["Date"] = df["Date"].apply(fix_year)
df["ISBN"] = df["ISBN"].apply(clean_isbn)

# Drop rows without Title+Author
df = df.dropna(subset=["Title", "Author"], how="all")

# Drop duplicates
df = df.drop_duplicates(subset=["Title", "Author", "ISBN"], keep="first")

# -------------------------------
# STEP 3: Select 17,000 books
# -------------------------------
books = df.dropna(subset=["Title", "Author"])
if len(books) > N_BOOKS:
    books = books.sample(N_BOOKS, random_state=42)

# -------------------------------
# STEP 4: Fetch Google Books metadata
# -------------------------------
def fetch_book_metadata(title, author):
    query = f"{title} {author}"
    params = {"q": query, "maxResults": 1}
    try:
        r = requests.get(API_URL, params=params, timeout=15)
        if r.status_code == 200:
            data = r.json()
            if data.get("totalItems", 0) > 0:
                info = data["items"][0].get("volumeInfo", {})
                return {
                    "cover_url": info.get("imageLinks", {}).get("thumbnail"),
                    "genre": ", ".join(info.get("categories", [])) if "categories" in info else None,
                    "description": info.get("description"),
                    "pageCount": info.get("pageCount"),
                    "rating": info.get("averageRating"),
                    "publishedDate": info.get("publishedDate"),
                }
    except Exception as e:
        print("⚠️ Error fetching:", query, e)
    return None

results = []

for i, row in enumerate(books.itertuples(index=False), start=1):
    meta = fetch_book_metadata(row.Title, row.Author)
    result = {
        "Title": row.Title,
        "Author": row.Author,
        "Edition": getattr(row, "Edition", None),
        "Imprint": getattr(row, "Imprint", None),
        "Date": getattr(row, "Date", None),
        "Call No.": getattr(row, "Call_No_", None) if hasattr(row, "Call_No_") else None,
        "ISBN": getattr(row, "ISBN", None),
    }
    if meta:
        result.update(meta)
        print(f"[{i}] ✅ {row.Title} — Found metadata (Cover: {meta.get('cover_url')})")
    else:
        print(f"[{i}] ❌ {row.Title} — No metadata found")

    results.append(result)

    # Prevent API quota issues
    time.sleep(0.1)

    # Show progress every 100 books
    if i % 100 == 0:
        print(f"--- Processed {i} books so far ---")

# -------------------------------
# STEP 5: Save final dataset
# -------------------------------
out_df = pd.DataFrame(results)
out_df.to_csv(OUTPUT_FILE, index=False)

print(f"\n🎉 Done! Saved {len(out_df)} books with metadata to {OUTPUT_FILE}")

/Users/kyii/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


FileNotFoundError: [Errno 2] No such file or directory: 'AU Library Books -ISBN-4-7-68 (1).csv'